In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

custom_llm = ChatOpenAI(
    model="minimax-local",
    api_key=os.getenv("CHAT_API_KEY"),
    base_url=os.getenv("CHAT_API_BASE_URL"),
    temperature=0,
    verbose=True
)

In [4]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.uuid import uuid7
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver


USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


@dataclass
class UserContext:
    user_id: str


@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account holder: {user['name']}\n"
            f"Type: {user['account_type']}\n"
            f"Balance: ${user['balance']}"
        )
    return "User not found"


agent = create_agent(
    custom_llm,
    tools=[get_account_info],
    context_schema=UserContext,
    checkpointer=InMemorySaver(),
    system_prompt="You are a financial assistant.",
)

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}
context = UserContext(user_id="user123")

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    config=config,
    context=context,
)
print(result["messages"][-1].content_blocks)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What was my balance again?"}]},
    config=config,
    context=context,
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': "Your current balance is **$5,000**.\n\nHere are your account details:\n- **Account holder:** Alice Johnson\n- **Account type:** Premium\n- **Balance:** $5,000\n\nIs there anything else you'd like to do with your account?"}]
[{'type': 'text', 'text': "Your balance is **$5,000** — same as before.\n\n- **Account holder:** Alice Johnson\n- **Account type:** Premium\n- **Balance:** $5,000\n\nLet me know if you'd like to do anything else!"}]


In [5]:
result["messages"]

[HumanMessage(content="What's my current balance?", additional_kwargs={}, response_metadata={}, id='91a48d31-3b43-4b21-9b70-6db67bf5a86f'),
 AIMessage(content="I'll look that up for you.\n\n", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 287, 'total_tokens': 334, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 16, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hosted_vllm/minimax-local', 'system_fingerprint': 'vllm-0.30.1rc1.dev493+gac9126e58-tp4-48484fca', 'id': 'chatcmpl-9f0ded0884ae8704', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a11ed1-e637-7060-b899-e92e32a26a9e-0', tool_calls=[{'name': 'get_account_info', 'args': {}, 'id': 'chatcmpl-tool-93f57ea94bc5094c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 287, 'output_tokens': 47, 'total_to